# VQCC - generate the current submission from pinned Apache-2.0 circuits

**Competition:** Google Quantum AI - Verifiable Quantum Circuit Compilation
(`google-verifiable-quantum-circuit-compilation`, competitionId 161954).
**Team:** *Nicks Seringian* (`nicks037`).

## What this notebook does, in order, with a gate at every step

| # | cell | what it does | the gate that stops us |
|---|---|---|---|
| 1 | config | resolves every path, prints the environment | the competition dataset must be found |
| 2 | fetch | copies the 18 members out of the dataset | every member must have a pinned SHA-256 and match it |
| 3 | verify | runs the competition's own three verifiers | all three verifiers must be FOUND and return `PASS` |
| 4 | package | writes `submission.zip` | every member re-hashed out of the packed archive |

## What it is honest about

The three circuits are **other people's Apache-2.0 public work**. This notebook
does not build them. It assembles them, pins every one of the 18 members with a
SHA-256, re-verifies them with the competition's own verifier, and writes the
archive.  Every gate fails closed: an unpinned member, a missing verifier and
a hash mismatch all stop the notebook before anything is packaged.
The upstream authors are named in NOTICE.md and in the receipts.


In [ ]:
# ==== CELL 1 - CONFIG + PATH RESOLUTION =====================================
import os, sys, json, glob, hashlib, subprocess, shutil

DATASET = "manwithacat/vqcc-apache-public-circuits"
OWNER, REPO, BRANCH = "EigenOS-Quantum", "eigenos-vqcc", "main"

# the three circuits: member path -> path inside the upstream dataset
LAYOUT = [
    ("invsqrt.kmb",                 "circuits/invsqrt/bce2948ff24e.kmb"),
    ("dqi.kmb",                     "circuits/dqi/3779a3431e9c.kmb"),
    ("dfthcsos/submission.npz",     "circuits/chemistry/289aafc7c5e7/submission.npz"),
] + [("dfthcsos/angle_k_%d.kmb" % k,
      "circuits/chemistry/289aafc7c5e7/angle_k_%d.kmb" % k) for k in range(15)]

# sha256 of EVERY member, pinned.  A member with no entry is a FAILURE.
EXPECTED = {
    "invsqrt.kmb":
        "bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47",
    "dqi.kmb":
        "3779a3431e9cb072bad980919236bd961ad3bcbd162d1757d51379316a103a2d",
    "dfthcsos/submission.npz":
        "64e7299eebd95fc37eb36de108e48cde33aaf6cdd17fa36e276e75f7d9af589c",
    "dfthcsos/angle_k_0.kmb":
        "8b698f14b339bd282e0332a1f105b41d60bfa5218d107c775db2532fbd88f774",
    "dfthcsos/angle_k_1.kmb":
        "0d16c5108e48e40931d87f1e3ee12d4486e932b341989ce909a47b032e5a7745",
    "dfthcsos/angle_k_10.kmb":
        "e2668bf5666ca1e49541bef7a255b2100a8a513188fadef3a8dd4e892cfe2094",
    "dfthcsos/angle_k_11.kmb":
        "5f69f385cb2040e7124e11e1cedb7cea0e0ffa6ad13abe81b2a337744cb9c2ac",
    "dfthcsos/angle_k_12.kmb":
        "f86995a0ea1990a11a43f1812f47e00a77658f71cc544d44bd5003e616111272",
    "dfthcsos/angle_k_13.kmb":
        "4390c8114e191b0c74ccd8b6302e5e0ded4e8cb412a3f146eea9887551658876",
    "dfthcsos/angle_k_14.kmb":
        "9e596deb120f20d1c92c350456ce9847314ba943a4d196e7942bac518a229f6c",
    "dfthcsos/angle_k_2.kmb":
        "b7b73491d3aeb59c70d023704df9f2ad69eb96c9d1f9e9920d33d60bd7852d11",
    "dfthcsos/angle_k_3.kmb":
        "c2437ad406d3772e961e7229580fba7b68c036b97ce36b4067c865e3af729f00",
    "dfthcsos/angle_k_4.kmb":
        "0df35ae1cb26d3bc09f022b2a397a8b07e5c322283d2c8cf0a53a4bb11b9dd47",
    "dfthcsos/angle_k_5.kmb":
        "a9a649997bf0696a659c8cbf9e778f92bee2816b6212febc898ed51450ae42c3",
    "dfthcsos/angle_k_6.kmb":
        "715673382c491b796865dbba235687eafebd16c745872d20f7ae1808f88a62ae",
    "dfthcsos/angle_k_7.kmb":
        "31e004f932d42750e1dda98abdf3e0097941cb1b1bbd353ed692e52098119cfb",
    "dfthcsos/angle_k_8.kmb":
        "046ac97b8c6ac9e1895a1da5dc89ea5df55bf487b1822d2ba11697c948145ff9",
    "dfthcsos/angle_k_9.kmb":
        "10c37a55ab902357cf60c5cf529de0eec9e9497a03f0b2ad7a9888d26d9c1e63",
}

def find_dataset_root():
    # 1. the competition data, if the reader attached it
    cands = ["/kaggle/input/datasets/" + DATASET, "/kaggle/input/" + DATASET,
             "/kaggle/input", "input/" + DATASET, "input", DATASET]
    # 2. the verifier tree, which usually sits next to the dataset
    for c in cands:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "circuits")):
            return c
    return None

DS = find_dataset_root()
print("dataset root :", DS)
print("python       :", sys.version.split()[0])

# the competition's verifiers: find the tree that has invsqrt/ dqi/ dfthcsos/
def find_verifiers():
    cands = ["/kaggle/input/verifiers", "/kaggle/input/competition/verifiers",
             "/kaggle/input/vqcc/verifiers", "verifiers"]
    for c in cands:
        if all(os.path.isdir(os.path.join(c, p)) for p in ("invsqrt", "dqi", "dfthcsos")):
            return c
    for root in ["/kaggle/input", "input", "."]:
        for hit in glob.glob(os.path.join(root, "**", "verifiers"), recursive=True):
            if all(os.path.isdir(os.path.join(hit, p)) for p in ("invsqrt", "dqi", "dfthcsos")):
                return hit
    return None

VERIFIERS = find_verifiers()
print("verifiers    :", VERIFIERS)
if DS is None:
    raise SystemExit("STOP: attach the dataset '" + DATASET + "' and re-run. "
                     "Everything below depends on it.")


In [ ]:
# ==== CELL 2 - FETCH + HASH EVERY MEMBER ====================================
WORK = "/kaggle/working/vqcc"
CIRCUITS = os.path.join(WORK, "circuits")
if os.path.isdir(CIRCUITS):
    shutil.rmtree(CIRCUITS)
os.makedirs(CIRCUITS)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

rows, bad = [], 0
for member, remote in LAYOUT:
    src = os.path.join(DS, remote)
    dst = os.path.join(CIRCUITS, member.replace("/", os.sep))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(src):
        print("MISSING IN DATASET", remote); bad += 1; continue
    shutil.copyfile(src, dst)
    got = sha256(dst)
    want = EXPECTED.get(member)
    if want is None:
        # Fail closed: an unpinned member is UNVERIFIED, and an unverified
        # member must never reach the archive.
        bad += 1
        os.remove(dst)
        rows.append({"member": member, "sha256": got, "bytes": 0, "ok": False,
                     "expected": None, "upstream": remote})
        print("%-5s %-30s no expected SHA-256 is pinned" % ("FAIL", member))
        continue
    ok = (got == want)
    if not ok:
        bad += 1
        os.remove(dst)
    rows.append({"member": member, "sha256": got, "bytes": os.path.getsize(dst)
                 if os.path.exists(dst) else 0, "ok": ok,
                 "expected": want, "upstream": remote})
    print("%-5s %-30s %s" % ("OK" if ok else "FAIL", member, got))

print()
print("%d of %d members OK, %d bad" % (len(rows) - bad, len(rows), bad))
if bad:
    raise SystemExit("STOP: a hash did not match. Do not package anything.")


In [ ]:
# ==== CELL 3 - VERIFY WITH THE COMPETITION'S OWN VERIFIERS ===================
# Nothing here re-implements the metric.  The competition's CLIs do the work.
import json

def run_verifier(pkg, script, args, label):
    path = os.path.join(VERIFIERS, pkg, script)
    if not os.path.exists(path):
        # Fail closed. A verifier that did not run is not a verifier that
        # passed. Packaging an unverified circuit would be a false receipt.
        print("STOP %s - verifier not found at %s" % (label, path))
        raise SystemExit(
            "STOP: the competition's " + label + " verifier was not found at " + path +
            ". Attach the competition data that carries the verifiers and re-run. "
            "This notebook will not package anything it has not verified.")
    cmd = [sys.executable, path] + args
    print("=== %s ===" % label)
    proc = subprocess.run(cmd, capture_output=True, text=True)
    try:
        rep = json.loads(proc.stdout)
    except Exception:
        print(proc.stdout[-1500:]); print(proc.stderr[-800:])
        raise SystemExit("STOP: %s did not return JSON" % label)
    print("   status=%s score=%s shots=%s" %
          (rep.get("status"), rep.get("score"), rep.get("verified_shots")))
    return rep

receipts = {}
receipts["invsqrt"] = run_verifier(
    "invsqrt", "verify_submission_invsqrt.py",
    [os.path.join(CIRCUITS, "invsqrt.kmb")], "invsqrt (fusion)")
receipts["dqi"] = run_verifier(
    "dqi", "verify_submission_dqi.py",
    [os.path.join(CIRCUITS, "dqi.kmb")], "dqi (combinatorial optimization)")
receipts["dfthcsos"] = run_verifier(
    "dfthcsos", "verify.py", [CIRCUITS], "dfthcsos (chemistry)")

failed = [k for k, v in receipts.items() if v is None or v.get("status") != "PASS"]
print()
if failed:
    raise SystemExit("STOP: verifier(s) did not PASS: " + ", ".join(failed))
print("all three verifiers returned PASS")

os.makedirs("/kaggle/working/receipts", exist_ok=True)
for k, v in receipts.items():
    if v is not None:
        with open("/kaggle/working/receipts/%s-verify.json" % k, "w") as fh:
            json.dump(v, fh, indent=2, default=str)


In [ ]:
# ==== CELL 4 - PACKAGE, AND RE-HASH OUT OF THE PACKED ARCHIVE ===============
import zipfile, tempfile

OUT = "/kaggle/working/submission.zip"
FIXED = (2026, 11, 2, 0, 0, 0)      # fixed timestamps -> byte-reproducible archive

if os.path.exists(OUT):
    os.remove(OUT)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for member, _ in LAYOUT:
        path = os.path.join(CIRCUITS, member.replace("/", os.sep))
        info = zipfile.ZipInfo(member, date_time=FIXED)
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        info.create_system = 3
        with open(path, "rb") as fh:
            z.writestr(info, fh.read(), compresslevel=9)

stage = tempfile.mkdtemp(prefix="vqcc_pack_")
with zipfile.ZipFile(OUT) as z:
    names = z.namelist()
    assert len(names) == 18, "expected 18 members, got %d" % len(names)
    z.extractall(stage)

manifest = {"members": []}
for member, _ in LAYOUT:
    p = os.path.join(stage, member.replace("/", os.sep))
    assert os.path.exists(p), "member missing from the packed archive: " + member
    manifest["members"].append({"member": member, "sha256": sha256(p),
                                 "bytes": os.path.getsize(p)})
shutil.rmtree(stage, ignore_errors=True)

manifest["archive_sha256"] = sha256(OUT)
manifest["archive_bytes"] = os.path.getsize(OUT)
with open("/kaggle/working/receipts/submission-manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)

print("archive        :", OUT)
print("archive bytes  :", manifest["archive_bytes"])
print("archive sha256 :", manifest["archive_sha256"])
print("members        :", len(manifest["members"]))
print()
print("UPLOAD THIS FILE: submission.zip")
print("Do not upload this notebook. Do not upload anything else.")
